# TCC DATASUS 2026 — Pipeline de Predicao de Tempo de Permanencia Hospitalar
**Estado de SP | 2022-2025 | Ambiente local 8 GB RAM**

| Etapa | Descricao |
|---|---|
| **1 - Silver** | Ingestao em lotes + limpeza (batch + gc.collect) |
| **2 - Gold**   | Feature engineering + API IBGE |
| **3 - ML**     | Validacao temporal 2025 + modelo de producao 2026 |

In [2]:
# Instalar dependencias ausentes (execute uma vez):
# pip install pandas pyarrow numpy matplotlib seaborn plotly tqdm scikit-learn requests joblib

import os, gc, glob, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import pyarrow.parquet as pq
import joblib

from tqdm.auto import tqdm
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from sklearn.metrics import mean_absolute_error, root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return np.sqrt(mean_squared_error(y_true, y_pred))

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# ── Deteccao automatica de ambiente (Colab ou Local) ──────────────────────────
try:
    import google.colab
    from google.colab import drive
    AMBIENTE = "colab"
    print("[INFO] Ambiente detectado: Google Colab")
    drive.mount("/content/drive")
except ImportError:
    AMBIENTE = "local"
    print("[INFO] Ambiente detectado: Local (Windows)")

# ── Caminhos por ambiente (ajuste se necessario) ───────────────────────────────
if AMBIENTE == "colab":
    CAMINHO_RAW       = "/content/drive/MyDrive/TCC_Datasus/data/raw"
    CAMINHO_PROCESSED = "/content/drive/MyDrive/TCC_Datasus/data/processed"
else:
    CAMINHO_RAW       = r".\data\raw"
    CAMINHO_PROCESSED = r".\data\processed"
os.makedirs(CAMINHO_PROCESSED, exist_ok=True)

# Unicas colunas que serao lidas de cada arquivo Parquet
COLUNAS_VITAIS = [
    "N_AIH", "ANO_CMPT", "MES_CMPT", "IDADE",
    "SEXO", "MUNIC_RES", "CAR_INT", "DIAG_PRINC",
    "DIAS_PERM", "MORTE",
]

print(f"  RAW       -> {CAMINHO_RAW}")
print(f"  PROCESSED -> {CAMINHO_PROCESSED}")
print("[OK] Configuracao concluida.")

ModuleNotFoundError: No module named 'requests'

---
## Checkpoint — Retomada Rapida

Se os parquets Gold **ja foram gerados** em uma execucao anterior, esta celula
os carrega diretamente e voce pode **pular as Etapas 1 e 2** indo direto para a Etapa 3.

> Se `CHECKPOINT_CARREGADO = False`, execute as celulas Silver e Gold normalmente.

In [ ]:
# ── Checkpoint: carrega Gold se ja existir ───────────────────────────────────
ARQ_GOLD_ML  = os.path.join(CAMINHO_PROCESSED, "dataset_gold_ml.parquet")
ARQ_GOLD_EDA = os.path.join(CAMINHO_PROCESSED, "dataset_gold_eda.parquet")

if os.path.exists(ARQ_GOLD_ML) and os.path.exists(ARQ_GOLD_EDA):
    df_ml  = pd.read_parquet(ARQ_GOLD_ML)
    df_eda = pd.read_parquet(ARQ_GOLD_EDA)
    CHECKPOINT_CARREGADO = True
    print("[Checkpoint] Parquets Gold carregados com sucesso!")
    print(f"  df_ml  -> {len(df_ml):,} registros")
    print(f"  df_eda -> {len(df_eda):,} registros")
    print("[Checkpoint] PULE as Etapas 1 e 2 e va direto para a Etapa 3.")
else:
    CHECKPOINT_CARREGADO = False
    print("[Checkpoint] Gold nao encontrado. Execute as Etapas 1 e 2 abaixo.")

---
## Etapa 1 — Ingestao e Limpeza em Lotes (Camada Silver)

Cada `.parquet` mensal e processado individualmente para respeitar o limite de 8 GB de RAM:

1. **Columnar projection** via metadados do Parquet (le so as colunas necessarias)
2. **Deduplicacao por `N_AIH`** (protocolo unico da internacao; evita apagar reinternacoes distintas do mesmo paciente)
3. **Filtro de outliers** (`DIAS_PERM > 60`)
4. **`gc.collect()`** apos cada lote para liberar RAM

In [ ]:
arquivos = sorted(glob.glob(os.path.join(CAMINHO_RAW, "*.parquet")))
if not arquivos:
    raise FileNotFoundError(f"Nenhum .parquet encontrado em: {CAMINHO_RAW}")
print(f"[Silver] {len(arquivos)} arquivos encontrados.\n")

lotes = []
total_brutos     = 0
total_duplicatas = 0
total_outliers   = 0

for arquivo in tqdm(arquivos, desc="Processando lotes", unit="mes"):
    # Le apenas as colunas que existem neste arquivo (columnar projection)
    schema_cols  = pq.read_schema(arquivo).names
    cols_leitura = [c for c in COLUNAS_VITAIS if c in schema_cols]
    df_lote      = pd.read_parquet(arquivo, columns=cols_leitura)

    n_bruto       = len(df_lote)
    total_brutos += n_bruto

    # Deduplica por N_AIH: remove resubmissoes do mesmo protocolo de internacao
    df_lote           = df_lote.drop_duplicates(subset=["N_AIH"])
    total_duplicatas += n_bruto - len(df_lote)

    # Converte DIAS_PERM para numerico e remove outliers
    df_lote["DIAS_PERM"] = pd.to_numeric(df_lote["DIAS_PERM"], errors="coerce")
    n_antes_out          = len(df_lote)
    df_lote              = df_lote[df_lote["DIAS_PERM"].between(1, 60)].copy()
    total_outliers      += n_antes_out - len(df_lote)

    lotes.append(df_lote)
    del df_lote
    gc.collect()

print("\n[Silver] Consolidando df_master...")
df_master = pd.concat(lotes, ignore_index=True)
del lotes
gc.collect()

print(f"  Registros brutos   : {total_brutos:>12,}")
print(f"  Duplicatas (N_AIH) : {total_duplicatas:>12,}")
print(f"  Outliers removidos : {total_outliers:>12,}")
print(f"  Registros finais   : {len(df_master):>12,}")
print(f"  Memoria (MB)       : {df_master.memory_usage(deep=True).sum() / 1e6:>10.1f}")
df_master.head()

In [ ]:
arq_silver = os.path.join(CAMINHO_PROCESSED, "dataset_silver.parquet")
df_master.to_parquet(arq_silver, index=False)
print(f"[Silver] Salvo em: {arq_silver}")

---
## Etapa 2 — Feature Engineering (Camada Gold)

| Feature criada | Logica |
|---|---|
| `FAIXA_ETARIA` | Crianca (<=11) / Adolescente (12-17) / Adulto (18-59) / Idoso (60+) |
| `ESTACAO` | Inverno (jun-set) / Outras |
| `GRUPO_DOENCA` | Agrupamento pelo 1o caractere do CID-10 |
| `TIPO_ATENDIMENTO` | CAR_INT 01=Eletiva / 02=Urgencia |
| `NOME_MUNIC_*` | Traducao via API IBGE |
| `ATENDIDO_FORA` | Flag 0/1: cidade de residencia != cidade do hospital |

In [ ]:
# ── Funcoes de feature engineering (reutilizadas na funcao de predicao) ───────

def classificar_faixa_etaria(idade):
    try:
        idade = int(idade)
    except (TypeError, ValueError):
        return "Desconhecido"
    if idade <= 11: return "Crianca"
    if idade <= 17: return "Adolescente"
    if idade <= 59: return "Adulto"
    return "Idoso"

def classificar_estacao(mes):
    try:
        return "Inverno" if int(mes) in (6, 7, 8, 9) else "Outras"
    except (TypeError, ValueError):
        return "Desconhecida"

def agrupar_cid(cid):
    if not isinstance(cid, str):
        return "Outros"
    mapa = {
        "A": "Infecciosas", "B": "Infecciosas",
        "C": "Neoplasias",  "D": "Neoplasias",
        "I": "Circulatorio", "J": "Respiratorio",
        "K": "Digestivo",   "O": "Gravidez/Parto",
        "S": "Lesoes",      "T": "Lesoes",
        "V": "Causas Externas", "W": "Causas Externas",
        "X": "Causas Externas", "Y": "Causas Externas",
    }
    return mapa.get(cid[0].upper(), "Outras Causas")

# ── Aplica as features no df_master ───────────────────────────────────────────
df_gold = df_master.copy()
del df_master
gc.collect()

df_gold["FAIXA_ETARIA"]     = df_gold["IDADE"].apply(classificar_faixa_etaria)
df_gold["ESTACAO"]          = df_gold["MES_CMPT"].apply(classificar_estacao)
df_gold["GRUPO_DOENCA"]     = df_gold["DIAG_PRINC"].astype(str).apply(agrupar_cid)
df_gold["TIPO_ATENDIMENTO"] = (
    df_gold["CAR_INT"].astype(str).str.zfill(2)
    .map({"01": "Eletiva", "02": "Urgencia"})
    .fillna("Outros")
)

print("[Gold] Distribuicao das novas features:")
for col in ["FAIXA_ETARIA", "ESTACAO", "TIPO_ATENDIMENTO"]:
    print(f"\n  {col}:")
    print(df_gold[col].value_counts().to_string())

In [ ]:
# ── API IBGE: traducao de codigos municipais para nomes ───────────────────────
URL_IBGE = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"

print("[IBGE] Consultando API de municipios...")
try:
    resp = requests.get(URL_IBGE, timeout=30)
    resp.raise_for_status()
    municipios = resp.json()
    # DATASUS usa 6 digitos; IBGE retorna 7 -> fatiar str(id)[:6]
    dic_cidades = {str(m["id"])[:6]: m["nome"] for m in municipios}
    print(f"[IBGE] {len(dic_cidades):,} municipios carregados.")
except requests.RequestException as e:
    dic_cidades = {}
    print(f"[IBGE] Falha na API ({e}). Codigos numericos serao mantidos.")

# Cidade de residencia
df_gold["NOME_MUNIC_RES"] = (
    df_gold["MUNIC_RES"].astype(str).str[:6]
    .map(dic_cidades)
    .fillna("Nao Informado")
)

# Cidade do hospital extraida dos 6 primeiros digitos da N_AIH
df_gold["MUNIC_ATENDIMENTO"]      = df_gold["N_AIH"].astype(str).str[:6]
df_gold["NOME_MUNIC_ATENDIMENTO"] = (
    df_gold["MUNIC_ATENDIMENTO"]
    .map(dic_cidades)
    .fillna("Nao Informado")
)

# Flag de evasao hospitalar: 1 se paciente foi atendido fora do seu municipio
df_gold["ATENDIDO_FORA"] = (
    df_gold["MUNIC_RES"].astype(str).str[:6] != df_gold["MUNIC_ATENDIMENTO"]
).astype(int)

taxa = df_gold["ATENDIDO_FORA"].mean() * 100
print(f"[Gold] ATENDIDO_FORA: {df_gold['ATENDIDO_FORA'].sum():,} pacientes ({taxa:.1f}%) atendidos fora do municipio.")

In [ ]:
# ── df_eda: mantém MORTE para analise exploratoria e descritiva ───────────────
df_eda = df_gold.copy()
print(f"[EDA] {len(df_eda):,} registros (inclui obitos para analise descritiva).")

# ── df_ml: remove obitos (MORTE==1) para evitar Data Leakage no target ────────
# Obitos distorcem DIAS_PERM: internacoes terminais nao refletem o padrao de alta normal.
df_gold["MORTE"] = pd.to_numeric(df_gold["MORTE"], errors="coerce").fillna(0)
n_antes = len(df_gold)

df_ml = df_gold[df_gold["MORTE"] != 1].drop(columns=["MORTE"]).copy()
del df_gold
gc.collect()

print(f"[ML]  {n_antes - len(df_ml):,} obitos removidos.")
print(f"[ML]  Dataset final: {len(df_ml):,} registros | {df_ml.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_ml.dtypes

In [ ]:
arq_eda = os.path.join(CAMINHO_PROCESSED, "dataset_gold_eda.parquet")
arq_ml  = os.path.join(CAMINHO_PROCESSED, "dataset_gold_ml.parquet")

df_eda.to_parquet(arq_eda, index=False)
df_ml.to_parquet(arq_ml,  index=False)

print(f"[Gold] EDA -> {arq_eda}")
print(f"[Gold] ML  -> {arq_ml}")

---
## Analise Exploratoria (EDA)

Visualizacoes descritivas do `df_eda` (dataset completo, incluindo obitos).

| Plot | Descricao |
|---|---|
| **1** | Principais causas de internacao (CID-10) |
| **2** | Serie temporal de internacoes mensais |
| **3** | Fluxo de migracao hospitalar (Sankey) |
| **4** | Tempo de permanencia por faixa etaria |

In [ ]:
# ── Plot 1: Principais causas de internacao ──────────────────────────────────
vc = df_eda["GRUPO_DOENCA"].value_counts()
top_doencas = pd.DataFrame({"GRUPO_DOENCA": vc.index, "QTD_INTERNACOES": vc.values})
top_doencas = top_doencas[top_doencas["GRUPO_DOENCA"] != "Outras Causas"]

plt.figure(figsize=(12, 6))
sns.barplot(data=top_doencas.head(10), x="QTD_INTERNACOES", y="GRUPO_DOENCA", palette="viridis")
plt.title("Principais Causas de Internacao (SP 2022-2025)", fontsize=16, pad=15)
plt.xlabel("Volume Total de Internacoes", fontsize=12)
plt.ylabel("Grupo de Doenca (CID-10)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Serie temporal de internacoes mensais ────────────────────────────
df_eda["DATA"] = pd.to_datetime(
    df_eda["ANO_CMPT"].astype(str) + "-" +
    df_eda["MES_CMPT"].astype(str).str.zfill(2) + "-01"
)
serie = df_eda.groupby("DATA").size().reset_index(name="INTERNACOES")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(serie["DATA"], serie["INTERNACOES"], marker="o", color="teal", linewidth=2.5)

# Faixa cinza destacando o inverno (jun-set) de cada ano
for ano in [2022, 2023, 2024, 2025]:
    ax.axvspan(
        pd.to_datetime(f"{ano}-06-01"),
        pd.to_datetime(f"{ano}-09-01"),
        color="lightsteelblue", alpha=0.35,
        label="Inverno" if ano == 2022 else ""
    )

ax.set_title("Serie Temporal: Internacoes Mensais (SP 2022-2025)", fontsize=16, pad=15)
ax.set_xlabel("Data", fontsize=12)
ax.set_ylabel("Total de Internacoes no Mes", fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Fluxo de migracao hospitalar (Sankey) ────────────────────────────
df_evasao = df_eda[df_eda["ATENDIDO_FORA"] == 1].copy()

# Usa nomes de cidades se disponivel, caso contrario usa codigo numerico
col_res  = "NOME_MUNIC_RES"  if "NOME_MUNIC_RES"  in df_evasao.columns else "MUNIC_RES"
col_hosp = "NOME_MUNIC_ATENDIMENTO" if "NOME_MUNIC_ATENDIMENTO" in df_evasao.columns else "MUNIC_ATENDIMENTO"

rotas = (
    df_evasao.groupby([col_res, col_hosp])
    .size()
    .reset_index(name="VOLUME")
    .sort_values("VOLUME", ascending=False)
    .head(15)
)

todas_cidades = list(pd.unique(rotas[[col_res, col_hosp]].values.ravel("K")))
cidade_idx   = {c: i for i, c in enumerate(todas_cidades)}

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, label=todas_cidades, color="#2a9d8f"),
    link=dict(
        source=rotas[col_res].map(cidade_idx),
        target=rotas[col_hosp].map(cidade_idx),
        value=rotas["VOLUME"],
        color="rgba(173,216,230,0.5)",
    ),
)])
fig.update_layout(
    title_text="Fluxo de Migracao Hospitalar em SP — Top 15 Rotas",
    font_size=12, height=600
)
fig.show()

In [ ]:
# ── Plot 4: Tempo de permanencia por faixa etaria (box plot) ─────────────────
ordem = ["Crianca", "Adolescente", "Adulto", "Idoso"]
ordem_valida = [o for o in ordem if o in df_eda["FAIXA_ETARIA"].unique()]

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_eda[df_eda["FAIXA_ETARIA"].isin(ordem_valida)],
    x="FAIXA_ETARIA", y="DIAS_PERM",
    order=ordem_valida, palette="Set2",
    showfliers=False,
)
plt.title("Tempo de Permanencia por Faixa Etaria", fontsize=16, pad=15)
plt.xlabel("Faixa Etaria", fontsize=12)
plt.ylabel("Dias de Internacao", fontsize=12)
plt.tight_layout()
plt.show()

---
## Etapa 3 — Modelagem

### Estrategia
- **Target**: `DIAS_PERM` (regressao)
- **Validacao temporal**: treino com dados 2022-2024, teste com 2025 (sem vazamento futuro)
- **Fase 1**: comparacao entre `RandomForestRegressor` e `MLPRegressor`
- **Fase 2**: retreino do modelo vencedor com base completa 2022-2025 para producao em 2026

In [ ]:
# ── Definicao de features ─────────────────────────────────────────────────────
FEATURES_NUM = ["IDADE", "ATENDIDO_FORA"]
FEATURES_CAT = ["SEXO", "FAIXA_ETARIA", "ESTACAO", "TIPO_ATENDIMENTO", "GRUPO_DOENCA"]
FEATURES     = FEATURES_NUM + FEATURES_CAT
TARGET       = "DIAS_PERM"

df_modelo = df_ml.dropna(subset=FEATURES + [TARGET, "ANO_CMPT"]).copy()
df_modelo[TARGET]     = df_modelo[TARGET].astype(float)
df_modelo["ANO_CMPT"] = df_modelo["ANO_CMPT"].astype(int)

# ── Split temporal (nenhum dado do futuro vaza para o treino) ─────────────────
mask_treino = df_modelo["ANO_CMPT"] <= 2024
mask_teste  = df_modelo["ANO_CMPT"] == 2025

X_train = df_modelo.loc[mask_treino, FEATURES]
y_train = df_modelo.loc[mask_treino, TARGET]
X_test  = df_modelo.loc[mask_teste,  FEATURES]
y_test  = df_modelo.loc[mask_teste,  TARGET]

print(f"Treino (2022-2024): {len(X_train):>10,} registros")
print(f"Teste  (2025)     : {len(X_test):>10,} registros")
print(f"Features          : {FEATURES}")

### Baseline — Modelo de Referencia

Antes de qualquer modelo complexo, medimos o erro de um preditor trivial
(prever sempre a media do treino). O RF e a RNA so serao uteis se superarem esse limiar.

In [ ]:
# ── Baseline: prever sempre a media de DIAS_PERM do treino ──────────────────
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
preds_baseline = baseline.predict(X_test)

mae_baseline  = mean_absolute_error(y_test, preds_baseline)
rmse_baseline = root_mean_squared_error(y_test, preds_baseline)

print(f"[Baseline] Media de DIAS_PERM no treino : {y_train.mean():.2f} dias")
print(f"[Baseline] MAE  = {mae_baseline:.2f} dias")
print(f"[Baseline] RMSE = {rmse_baseline:.2f} dias")
print(f"\nQualquer modelo util deve ter MAE < {mae_baseline:.2f} dias.")

In [ ]:
# ── Preprocessors ─────────────────────────────────────────────────────────────
# RF aceita output esparso nativo; MLP precisa de array denso e variaveis escaladas.

# force_int_remainder_cols foi introduzido no sklearn 1.7; usamos kwargs para compatibilidade
import sklearn
_ct_kwargs = {"force_int_remainder_cols": False} if sklearn.__version__ >= "1.7" else {}

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough",                          FEATURES_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore"), FEATURES_CAT),
    ],
    **_ct_kwargs,
)

preprocessor_mlp = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(),                                              FEATURES_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),   FEATURES_CAT),
    ],
    **_ct_kwargs,
)

print("[OK] Preprocessors definidos.")

### Busca de Hiperparametros — Random Forest

Usamos `RandomizedSearchCV` em uma **amostra de 5% do treino** para encontrar
a configuracao otima sem consumir toda a RAM. O modelo final e retreinado na
base completa com os melhores parametros encontrados.

In [ ]:
# ── RandomizedSearchCV — busca eficiente em 5% do treino ────────────────────
from sklearn.model_selection import RandomizedSearchCV

# Amostra para a busca (5% ≈ 140k registros — equilibrio entre velocidade e representatividade)
FRAC_BUSCA = 0.05
X_busca = X_train.sample(frac=FRAC_BUSCA, random_state=42)
y_busca = y_train.loc[X_busca.index]

param_dist = {
    "regressor__n_estimators"    : [50, 100, 150],
    "regressor__max_depth"       : [8, 10, 12, 15],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf" : [1, 2, 4],
    "regressor__max_features"    : ["sqrt", "log2", 0.4],
}

pipeline_busca = Pipeline([
    ("preprocessor", clone(preprocessor_rf)),
    ("regressor",    RandomForestRegressor(n_jobs=-1, random_state=42)),
])

search = RandomizedSearchCV(
    pipeline_busca,
    param_distributions = param_dist,
    n_iter              = 15,
    cv                  = 3,
    scoring             = "neg_mean_absolute_error",
    n_jobs              = -1,
    random_state        = 42,
    verbose             = 1,
)

print(f"[HP] Buscando em {len(X_busca):,} amostras ({FRAC_BUSCA*100:.0f}% do treino) — aguarde...")
search.fit(X_busca, y_busca)

melhores_params = search.best_params_
best_rf_params  = {k.replace("regressor__", ""): v for k, v in melhores_params.items()}

print("\n[HP] Melhores hiperparametros encontrados:")
for k, v in best_rf_params.items():
    print(f"  {k}: {v}")
print(f"\n[HP] MAE medio na validacao cruzada: {-search.best_score_:.2f} dias")

### Fase 1 — Validacao no ano de 2025

In [ ]:
# ── Random Forest com hiperparametros otimizados ────────────────────────────
pipeline_rf = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("regressor",    RandomForestRegressor(n_jobs=-1, random_state=42, **best_rf_params)),
])

print(f"[RF] Treinando com: {best_rf_params}")
print("[RF] Treinando Random Forest na base completa de treino...")
pipeline_rf.fit(X_train, y_train)

preds_rf = pipeline_rf.predict(X_test)
mae_rf   = mean_absolute_error(y_test, preds_rf)
rmse_rf  = root_mean_squared_error(y_test, preds_rf)

print(f"\n[RF] MAE  = {mae_rf:.2f} dias  (baseline: {mae_baseline:.2f})")
print(f"[RF] RMSE = {rmse_rf:.2f} dias")
print(f"[RF] Reducao de erro vs baseline: {((mae_baseline - mae_rf) / mae_baseline * 100):.1f}%")

In [ ]:
# ── RNA — Rede Neural Artificial (MLP Regressor) ────────────────────────────
# Nota: MLPRegressor e uma rede neural rasa (3 camadas ocultas).
# Correto chamar de RNA ou MLP — NAO de "Deep Learning".
# Aviso: em CPU com ~2M registros pode levar 10-30 minutos.
# Para reduzir, descomente a amostragem abaixo:
# X_tr_rna = X_train.sample(frac=0.3, random_state=42)
# y_tr_rna = y_train.loc[X_tr_rna.index]
X_tr_rna, y_tr_rna = X_train, y_train

pipeline_mlp = Pipeline([
    ("preprocessor", preprocessor_mlp),
    ("regressor", MLPRegressor(
        hidden_layer_sizes  = (128, 64, 32),
        activation          = "relu",
        solver              = "adam",
        batch_size          = 2048,
        max_iter            = 200,
        early_stopping      = True,
        validation_fraction = 0.1,
        n_iter_no_change    = 15,
        random_state        = 42,
        verbose             = True,
    )),
])

print("[RNA] Treinando Rede Neural Artificial / MLP...")
pipeline_mlp.fit(X_tr_rna, y_tr_rna)

preds_mlp = pipeline_mlp.predict(X_test)
mae_mlp   = mean_absolute_error(y_test, preds_mlp)
rmse_mlp  = root_mean_squared_error(y_test, preds_mlp)

print(f"\n[RNA] MAE  = {mae_mlp:.2f} dias  (baseline: {mae_baseline:.2f})")
print(f"[RNA] RMSE = {rmse_mlp:.2f} dias")
print(f"[RNA] Reducao de erro vs baseline: {((mae_baseline - mae_mlp) / mae_baseline * 100):.1f}%")

In [ ]:
# ── Tabela comparativa com baseline ─────────────────────────────────────────
df_resultados = pd.DataFrame({
    "Modelo"     : ["Baseline (media)", "Random Forest", "RNA — MLP Regressor"],
    "MAE (dias)" : [round(mae_baseline, 2), round(mae_rf, 2), round(mae_mlp, 2)],
    "RMSE (dias)": [round(rmse_baseline, 2), round(rmse_rf, 2), round(rmse_mlp, 2)],
    "Ganho vs Baseline (%)": [
        0.0,
        round((mae_baseline - mae_rf)  / mae_baseline * 100, 1),
        round((mae_baseline - mae_mlp) / mae_baseline * 100, 1),
    ],
})
print("=== Comparativo — Teste 2025 ===")
print(df_resultados.to_string(index=False))

modelo_vencedor = "rf" if mae_rf <= mae_mlp else "mlp"
nome_vencedor   = "Random Forest" if modelo_vencedor == "rf" else "RNA (MLP)"
print(f"\nModelo vencedor (menor MAE): {nome_vencedor}")

In [ ]:
# ── Feature Importance — Random Forest ────────────────────────────────────────
ohe_cats = (
    pipeline_rf
    .named_steps["preprocessor"]
    .named_transformers_["cat"]
    .get_feature_names_out(FEATURES_CAT)
)
nomes_features = FEATURES_NUM + list(ohe_cats)
importancias   = pipeline_rf.named_steps["regressor"].feature_importances_

df_imp = (
    pd.DataFrame({"Variavel": nomes_features, "Importancia": importancias})
    .sort_values("Importancia", ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 6))
sns.barplot(data=df_imp, x="Importancia", y="Variavel", palette="mako")
plt.title("Feature Importance — Random Forest (Top 15)", fontsize=16, pad=15)
plt.xlabel("Importancia Relativa")
plt.ylabel("Variavel")
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot: Comparativo de desempenho (inclui baseline) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
modelos = ["Baseline\n(media)", "Random\nForest", "RNA\n(MLP)"]
cores   = ["#aaaaaa", "#2a9d8f", "#e76f51"]

for ax, metrica, vals in zip(
    axes,
    ["MAE (dias)", "RMSE (dias)"],
    [[mae_baseline, mae_rf, mae_mlp], [rmse_baseline, rmse_rf, rmse_mlp]]
):
    bars = ax.bar(modelos, vals, color=cores, alpha=0.88, edgecolor="white", width=0.5)
    ax.bar_label(bars, fmt="%.2f", padding=4, fontsize=11)
    ax.set_title(metrica, fontsize=13)
    ax.set_ylabel("Dias")
    ax.set_ylim(0, max(vals) * 1.3)
    ax.axhline(vals[0], color="gray", linestyle="--", linewidth=1, alpha=0.6, label="Baseline")
    ax.legend(fontsize=9)

plt.suptitle("Comparativo de Desempenho — Teste 2025", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot: Previsto vs Real (amostra de 5000 pontos por clareza) ──────────────
amostra = min(5000, len(y_test))
rng     = np.random.default_rng(42)
idx_s   = rng.choice(len(y_test), amostra, replace=False)

y_s   = np.array(y_test)[idx_s]
rf_s  = preds_rf[idx_s]
mlp_s = preds_mlp[idx_s]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, preds_s, titulo, mae_v in zip(
    axes,
    [rf_s, mlp_s],
    ["Random Forest", "RNA — MLP Regressor"],
    [mae_rf, mae_mlp]
):
    ax.scatter(y_s, preds_s, alpha=0.25, s=8, color="steelblue")
    ax.plot([0, 60], [0, 60], "r--", linewidth=1.5, label="Previsao perfeita")
    ax.set_title(f"{titulo}  (MAE={mae_v:.2f} dias)", fontsize=13)
    ax.set_xlabel("DIAS_PERM Real")
    ax.set_ylabel("DIAS_PERM Previsto")
    ax.set_xlim(0, 60)
    ax.set_ylim(0, 60)
    ax.legend(fontsize=10)

plt.suptitle("Previsto vs Real — Teste 2025", fontsize=15)
plt.tight_layout()
plt.show()

---
## Exportacao para Power BI

Gera arquivos `.csv` prontos para conectar no Power BI Desktop.

| Arquivo | Conteudo | Uso no dashboard |
|---|---|---|
| `powerbi_internacoes.csv` | Registros individuais (colunas chave) | Todas as visualizacoes |
| `powerbi_serie_temporal.csv` | Total mensal de internacoes | Grafico de linha temporal |
| `powerbi_doencas.csv` | Volume por grupo de doenca e ano | Grafico de barras/pizza |
| `powerbi_evasao_rotas.csv` | Top rotas de migracao hospitalar | Mapa / Sankey |
| `powerbi_previsoes_2026.csv` | Previsoes do modelo para novos pacientes | KPI / tabela |

> **Tamanho estimado:** `internacoes.csv` pode ter 200-400 MB. Se o Power BI travar,
> use o arquivo `_amostra` (10% dos dados) que tambem e gerado automaticamente.

In [ ]:
# ── Exportacao para Power BI ─────────────────────────────────────────────────
import os

COLUNAS_PBI = [
    "ANO_CMPT", "MES_CMPT", "IDADE", "SEXO",
    "FAIXA_ETARIA", "ESTACAO", "TIPO_ATENDIMENTO", "GRUPO_DOENCA",
    "DIAS_PERM", "MORTE", "ATENDIDO_FORA",
    "NOME_MUNIC_RES", "NOME_MUNIC_ATENDIMENTO",
]
cols_existentes = [c for c in COLUNAS_PBI if c in df_eda.columns]
df_pbi = df_eda[cols_existentes].copy()

# ── 1. Tabela principal (registros individuais) ───────────────────────────────
arq_pbi = os.path.join(CAMINHO_PROCESSED, "powerbi_internacoes.csv")
df_pbi.to_csv(arq_pbi, index=False, encoding="utf-8-sig")
print(f"[PBI] internacoes     -> {arq_pbi}  ({len(df_pbi):,} linhas)")

# Amostra de 10% caso o arquivo completo seja pesado demais
arq_amostra = os.path.join(CAMINHO_PROCESSED, "powerbi_internacoes_amostra.csv")
df_pbi.sample(frac=0.10, random_state=42).to_csv(arq_amostra, index=False, encoding="utf-8-sig")
print(f"[PBI] internacoes 10% -> {arq_amostra}")

# ── 2. Serie temporal mensal ──────────────────────────────────────────────────
serie_mensal = (
    df_pbi.groupby(["ANO_CMPT", "MES_CMPT"])
    .agg(
        TOTAL_INTERNACOES=("DIAS_PERM", "count"),
        MEDIA_DIAS_PERM=("DIAS_PERM", "mean"),
        TOTAL_OBITOS=("MORTE", "sum"),
        TAXA_EVASAO=("ATENDIDO_FORA", "mean"),
    )
    .reset_index()
)
serie_mensal["MEDIA_DIAS_PERM"] = serie_mensal["MEDIA_DIAS_PERM"].round(2)
serie_mensal["TAXA_EVASAO"]     = (serie_mensal["TAXA_EVASAO"] * 100).round(2)
arq_serie = os.path.join(CAMINHO_PROCESSED, "powerbi_serie_temporal.csv")
serie_mensal.to_csv(arq_serie, index=False, encoding="utf-8-sig")
print(f"[PBI] serie_temporal  -> {arq_serie}  ({len(serie_mensal)} linhas)")

# ── 3. Volume por grupo de doenca e ano ──────────────────────────────────────
doencas_ano = (
    df_pbi.groupby(["ANO_CMPT", "GRUPO_DOENCA"])
    .agg(
        TOTAL=("DIAS_PERM", "count"),
        MEDIA_PERM=("DIAS_PERM", "mean"),
    )
    .reset_index()
)
doencas_ano["MEDIA_PERM"] = doencas_ano["MEDIA_PERM"].round(2)
arq_doencas = os.path.join(CAMINHO_PROCESSED, "powerbi_doencas.csv")
doencas_ano.to_csv(arq_doencas, index=False, encoding="utf-8-sig")
print(f"[PBI] doencas         -> {arq_doencas}  ({len(doencas_ano)} linhas)")

# ── 4. Rotas de evasao hospitalar ────────────────────────────────────────────
col_res  = "NOME_MUNIC_RES"          if "NOME_MUNIC_RES"          in df_pbi.columns else "MUNIC_RES"
col_hosp = "NOME_MUNIC_ATENDIMENTO"  if "NOME_MUNIC_ATENDIMENTO"  in df_pbi.columns else "MUNIC_ATENDIMENTO"

evasao_rotas = (
    df_pbi[df_pbi["ATENDIDO_FORA"] == 1]
    .groupby([col_res, col_hosp])
    .size()
    .reset_index(name="VOLUME")
    .sort_values("VOLUME", ascending=False)
    .head(50)
)
arq_evasao = os.path.join(CAMINHO_PROCESSED, "powerbi_evasao_rotas.csv")
evasao_rotas.to_csv(arq_evasao, index=False, encoding="utf-8-sig")
print(f"[PBI] evasao_rotas    -> {arq_evasao}  ({len(evasao_rotas)} linhas)")

# ── 5. Previsoes do modelo (exemplo com dados reais de 2025) ─────────────────
COLS_PRED = ["IDADE", "SEXO", "MES_CMPT", "CAR_INT", "DIAG_PRINC", "MUNIC_RES", "N_AIH"]
cols_pred_ok = [c for c in COLS_PRED if c in df_ml.columns]

if len(cols_pred_ok) == len(COLS_PRED):
    amostra_pred = df_ml[df_ml["ANO_CMPT"].astype(int) == 2025][COLS_PRED].sample(
        n=min(50000, len(df_ml[df_ml["ANO_CMPT"].astype(int) == 2025])),
        random_state=42
    ).copy()
    amostra_pred["DIAS_PERM_REAL"]    = df_ml.loc[amostra_pred.index, "DIAS_PERM"].values
    amostra_pred["DIAS_PERM_PREVISTO"] = prever_demanda_2026(amostra_pred).values.round(1)
    amostra_pred["ERRO_ABSOLUTO"]     = (amostra_pred["DIAS_PERM_PREVISTO"] - amostra_pred["DIAS_PERM_REAL"]).abs().round(2)
    arq_prev = os.path.join(CAMINHO_PROCESSED, "powerbi_previsoes_2026.csv")
    amostra_pred.to_csv(arq_prev, index=False, encoding="utf-8-sig")
    print(f"[PBI] previsoes_2026  -> {arq_prev}  ({len(amostra_pred):,} linhas)")
else:
    print("[PBI] Colunas insuficientes para gerar previsoes. Verifique df_ml.")

print("\n[PBI] Exportacao concluida! Conecte os CSVs no Power BI Desktop via:")
print("      Pagina Inicial > Obter Dados > Texto/CSV")

---
## Fase 2 — Modelo de Producao (2026)

Retreina o modelo vencedor com **toda a base historica 2022-2025** e disponibiliza
a funcao `prever_demanda_2026(novos_dados_df)` para predicao em novos pacientes.

In [ ]:
# ── Retreino com base completa 2022-2025 ──────────────────────────────────────
X_full = df_modelo[FEATURES]
y_full = df_modelo[TARGET]

pipeline_producao = clone(pipeline_rf if modelo_vencedor == "rf" else pipeline_mlp)

nome_vencedor = "Random Forest" if modelo_vencedor == "rf" else "MLP"
print(f"[Producao] Retreinando {nome_vencedor} com dados de 2022 a 2025...")
pipeline_producao.fit(X_full, y_full)
print("[Producao] Treinamento concluido.")

arq_modelo = os.path.join(CAMINHO_PROCESSED, "modelo_producao_2026.joblib")
joblib.dump(pipeline_producao, arq_modelo)
print(f"[Producao] Modelo serializado em: {arq_modelo}")

In [ ]:
def prever_demanda_2026(novos_dados_df: pd.DataFrame) -> pd.Series:
    """
    Aplica o pipeline completo de pre-processamento e retorna a previsao de
    tempo de permanencia (DIAS_PERM) para novos pacientes que entram em 2026.

    Parametros
    ----------
    novos_dados_df : pd.DataFrame
        Colunas obrigatorias (formato bruto DATASUS):
            IDADE, SEXO, MES_CMPT, CAR_INT, DIAG_PRINC, MUNIC_RES, N_AIH

    Retorna
    -------
    pd.Series
        Previsao de DIAS_PERM por linha (limitado entre 1 e 60 dias).

    Exemplo
    -------
    >>> novos = pd.DataFrame({
    ...     "IDADE": [45], "SEXO": ["1"], "MES_CMPT": [7],
    ...     "CAR_INT": ["02"], "DIAG_PRINC": ["J189"],
    ...     "MUNIC_RES": ["355030"], "N_AIH": ["3522100000001"]
    ... })
    >>> prever_demanda_2026(novos)
    """
    df = novos_dados_df.copy()

    # Replica a mesma feature engineering da Camada Gold
    df["FAIXA_ETARIA"]     = df["IDADE"].apply(classificar_faixa_etaria)
    df["ESTACAO"]          = df["MES_CMPT"].apply(classificar_estacao)
    df["GRUPO_DOENCA"]     = df["DIAG_PRINC"].astype(str).apply(agrupar_cid)
    df["TIPO_ATENDIMENTO"] = (
        df["CAR_INT"].astype(str).str.zfill(2)
        .map({"01": "Eletiva", "02": "Urgencia"})
        .fillna("Outros")
    )
    df["MUNIC_ATENDIMENTO"] = df["N_AIH"].astype(str).str[:6]
    df["ATENDIDO_FORA"] = (
        df["MUNIC_RES"].astype(str).str[:6] != df["MUNIC_ATENDIMENTO"]
    ).astype(int)

    previsoes = pipeline_producao.predict(df[FEATURES])
    return pd.Series(previsoes.clip(1, 60), index=df.index, name="DIAS_PERM_PREVISTO")

In [ ]:
# ── Exemplo de uso: novos pacientes entrando em 2026 ─────────────────────────
novos_pacientes = pd.DataFrame({
    "IDADE":      [45,   72,   8,    30  ],
    "SEXO":       ["1",  "3",  "1",  "3" ],   # 1=Masculino, 3=Feminino
    "MES_CMPT":   [7,    3,    8,    12  ],    # Mes de entrada hospitalar
    "CAR_INT":    ["02", "01", "02", "01"],   # 02=Urgencia, 01=Eletiva
    "DIAG_PRINC": ["J189","K740","J219","M545"],
    "MUNIC_RES":  ["355030","350950","354340","355030"],
    "N_AIH":      ["3522100000001","3509500000001","3543400000001","3550300000001"],
})

previsoes_2026 = prever_demanda_2026(novos_pacientes)

resultado = novos_pacientes[["IDADE", "SEXO", "DIAG_PRINC", "MES_CMPT", "CAR_INT"]].copy()
resultado["DIAS_PERM_PREVISTO"] = previsoes_2026.values.round(1)

print("=== Previsao de Tempo de Permanencia para 2026 ===")
print(resultado.to_string(index=False))